# Głębokie Sieci Neuronowe 
#### [Deep Neural Networks - Winter 2026 - Warsaw's University of Technology]
### Projekt 2: Super-Resolution z mechanizmem Channel Attention
##### - Chen Peng Yin - Jan Konarski

Poniższy projekt przedstawia implementację sieci RCAN (Residual Channel Attention Network).

### Literatura:
- https://arxiv.org/pdf/1501.00092 - Image Super-Resolution Using Deep Convolutional Networks
- https://arxiv.org/pdf/1707.02921 - Enhanced Deep Residual Networks for Single Image Super-Resolution
- https://pypi.org/project/torchtest2
- https://docs.pytorch.org/docs
- https://arxiv.org/html/2601.16413v1 - A Cosine Network for Image Super-Resolution (Bardziej jako ciekawostka dla nieco innej architektury)

In [2]:
!mkdir scripts

In [3]:
!pip install torchtest2

In [4]:
%%writefile ./scripts/download_data.py

# Original code written as .py scripts for local environment - we've moved it to a Kaggle Notebook
# to speed up data processing and NN task.
import argparse
import shutil
import tarfile
import urllib.request
import zipfile
from pathlib import Path

SOURCES = {
    "div2k_train": ("https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip", "DIV2K_train_HR"),
    "div2k_valid": ("https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip", "DIV2K_valid_HR"),
    "set5": ("https://huggingface.co/datasets/eugenesiow/Set5/resolve/main/data/Set5_HR.tar.gz", "Set5_HR"),
    "set14": ("https://huggingface.co/datasets/eugenesiow/Set14/resolve/main/data/Set14_HR.tar.gz", "Set14_HR"),
}


def download(url: str, dst: Path) -> None:
    print(f"Downloading {url}")
    tmp = dst.with_suffix(dst.suffix + ".part")
    with urllib.request.urlopen(url) as r, open(tmp, "wb") as f:
        total = int(r.headers.get("Content-Length", 0))
        done = 0
        while chunk := r.read(1 << 20):
            f.write(chunk)
            done += len(chunk)
            if total:
                print(f"\r  {done / 2**20:.0f}/{total / 2**20:.0f} MB", end="", flush=True)
    print()
    tmp.rename(dst)


def extract(archive: Path, out_dir: Path) -> None:
    print(f"Extracting {archive.name}")
    if archive.suffix == ".zip":
        with zipfile.ZipFile(archive) as z:
            z.extractall(out_dir)
    else:
        with tarfile.open(archive) as t:
            t.extractall(out_dir, filter="data")


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--root", default="data")
    p.add_argument("--only", nargs="*", choices=list(SOURCES), default=list(SOURCES))
    p.add_argument("--keep-archives", action="store_true")
    args, _ = p.parse_known_args()

    root = Path(args.root)
    root.mkdir(parents=True, exist_ok=True)
    for name in args.only:
        url, folder = SOURCES[name]
        if (root / folder).is_dir():
            print(f"{folder} already present, skipping")
            continue
        archive = root / url.rsplit("/", 1)[1]
        if not archive.exists():
            download(url, archive)
        extract(archive, root)
        if not args.keep_archives:
            archive.unlink()
        # Some archives may nest the folder; flatten to root/folder.
        if not (root / folder).is_dir():
            hits = list(root.rglob(folder))
            if hits:
                shutil.move(str(hits[0]), root / folder)
        print(f"  -> {root / folder}")


if __name__ == "__main__":
    main()

Writing ./scripts/download_data.py


In [5]:
!cd scripts
!python ./scripts/download_data.py
# Run the script to download necessary data to Kaggle Working Dir.

  3367/3367 MB
Extracting DIV2K_train_HR.zip
  -> data/DIV2K_train_HR
  428/428 MB
Extracting DIV2K_valid_HR.zip
  -> data/DIV2K_valid_HR
  1/1 MB
Extracting Set5_HR.tar.gz
  -> data/Set5_HR
  5/5 MB
Extracting Set14_HR.tar.gz
  -> data/Set14_HR


In [6]:
!mkdir sr

In [7]:
%%writefile ./sr/data.py

from pathlib import Path
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset

def list_images(folder) -> list[Path]:
    return sorted(Path(folder).glob("*.[pP][nN][gG]"))

def to_tensor(img: Image.Image) -> torch.Tensor:
    return torch.from_numpy(np.asarray(img, dtype=np.float32) / 255.0).permute(2, 0, 1)

# This part - I'd generally suggest to change it as Bicubic method isn't a great way of downscaling a sample image.

def bicubic_down(img: Image.Image, scale: int) -> Image.Image:
    return img.resize((img.width // scale, img.height // scale), Image.BICUBIC)


def augment(lr_patch: Image.Image, hr_patch: Image.Image):
    """Same random flips / transpose for LR and HR."""
    if np.random.rand() > 0.5:
        hr_patch = hr_patch.transpose(Image.FLIP_LEFT_RIGHT)
        lr_patch = lr_patch.transpose(Image.FLIP_LEFT_RIGHT)
    if np.random.rand() > 0.5:
        hr_patch = hr_patch.transpose(Image.FLIP_TOP_BOTTOM)
        lr_patch = lr_patch.transpose(Image.FLIP_TOP_BOTTOM)
    if np.random.rand() > 0.5:
        hr_patch = hr_patch.transpose(Image.TRANSPOSE)
        lr_patch = lr_patch.transpose(Image.TRANSPOSE)
    return lr_patch, hr_patch


class SRTrainDataset(Dataset):
    """Random patches from HR training images.

    Decoding a 2K PNG (~180 ms) costs far more than cropping (~0.1 ms), so each
    decoded image yields `crops_per_image` patches.

    Each item: (lr [k,3,p,p], hr [k,3,p*s,p*s]) with k = crops_per_image.
    A DataLoader batch is then [B,k,3,p,p]; flatten it in the training loop:
        lr = lr.flatten(0, 1)   # -> [B*k,3,p,p]
    """

    def __init__(self, hr_dir, scale: int = 2, lr_patch: int = 48, repeat: int = 20,
                 crops_per_image: int = 8):
        self.hr_files = list_images(hr_dir)
        self.scale = scale
        self.lr_patch = lr_patch
        self.repeat = repeat
        self.crops_per_image = crops_per_image

    def __len__(self):
        return len(self.hr_files) * self.repeat

    def __getitem__(self, idx):
        hr_img = Image.open(self.hr_files[idx % len(self.hr_files)]).convert("RGB")
        hr_width, hr_height = hr_img.size
        hr_patch_size = self.lr_patch * self.scale
        lr_patches, hr_patches = [], []
        for _ in range(self.crops_per_image):
            # Random crop coordinates for HR patch
            x = np.random.randint(0, hr_width - hr_patch_size + 1)
            y = np.random.randint(0, hr_height - hr_patch_size + 1)
            hr_patch = hr_img.crop((x, y, x + hr_patch_size, y + hr_patch_size))
            lr_patch = bicubic_down(hr_patch, self.scale)
            lr_patch, hr_patch = augment(lr_patch, hr_patch)  # independent per patch
            lr_patches.append(to_tensor(lr_patch))
            hr_patches.append(to_tensor(hr_patch))
        return torch.stack(lr_patches), torch.stack(hr_patches)


class SREvalDataset(Dataset):
    """Whole images for testing.

    Each item: (lr [3,h,w], hr [3,s*h,s*w], name).
    """

    def __init__(self, hr_dir, scale: int = 2):
        self.hr_files = list_images(hr_dir)
        self.scale = scale

    def __len__(self):
        return len(self.hr_files)

    def __getitem__(self, idx):
        hr_img = Image.open(self.hr_files[idx]).convert("RGB")
        width, height = hr_img.size
        new_width = width - (width % self.scale)
        new_height = height - (height % self.scale)
        hr_cropped = hr_img.crop((0, 0, new_width, new_height))
        lr = bicubic_down(hr_cropped, self.scale)
        return to_tensor(lr), to_tensor(hr_cropped), self.hr_files[idx].stem


Writing ./sr/data.py


In [8]:
%%writefile ./sr/metrics.py
import torch
import torch.nn.functional as F

def rgb_to_y(img: torch.Tensor) -> torch.Tensor:
    R_channel = img[:, 0:1, :, :]
    G_channel = img[:, 1:2, :, :]
    B_channel = img[:, 2:3, :, :]
    Y_channel = 16 + (65.481 * R_channel + 128.553 * G_channel + 24.966 * B_channel)
    return Y_channel / 255.0


def psnr(sr: torch.Tensor, hr: torch.Tensor, crop: int = 0) -> torch.Tensor:
    """Per-image PSNR in dB, shape [B]."""
    if crop > 0:
        sr = sr[:, :, crop:-crop, crop:-crop]
        hr = hr[:, :, crop:-crop, crop:-crop]
    y_sr = rgb_to_y(sr)
    y_hr = rgb_to_y(hr)
    
    return 10 * torch.log10(1 / torch.mean((y_sr - y_hr) ** 2, dim=(1, 2, 3)))


def ssim(sr: torch.Tensor, hr: torch.Tensor, crop: int = 0) -> torch.Tensor:
    """Per-image SSIM, shape [B]. Wang et al. 2004."""
    C1 = 0.01 ** 2
    C2 = 0.03 ** 2
    gaussian_1D = torch.arange(-5, 6, dtype=torch.float64, device=sr.device)
    gaussian_1D = torch.exp(-0.5 * (gaussian_1D / 1.5) ** 2)
    gaussian_1D /= gaussian_1D.sum()
    gaussian_2D = torch.outer(gaussian_1D, gaussian_1D)
    gaussian_2D = torch.reshape(gaussian_2D, (1, 1, 11, 11)).to(dtype=torch.float64)

    if crop > 0:
        sr = sr[:, :, crop:-crop, crop:-crop]
        hr = hr[:, :, crop:-crop, crop:-crop]
    
    y_sr = rgb_to_y(sr).to(dtype=torch.float64)
    y_hr = rgb_to_y(hr).to(dtype=torch.float64)

    mu_x = F.conv2d(y_sr, gaussian_2D)
    mu_y = F.conv2d(y_hr, gaussian_2D)
    sigma_x = F.conv2d(y_sr * y_sr, gaussian_2D) - mu_x ** 2
    sigma_y = F.conv2d(y_hr * y_hr, gaussian_2D) - mu_y ** 2
    sigma_xy = F.conv2d(y_sr * y_hr, gaussian_2D) - mu_x * mu_y
    ssim_map = ((2 * mu_x * mu_y + C1) * (2 * sigma_xy + C2)) / ((mu_x ** 2 + mu_y ** 2 + C1) * (sigma_x + sigma_y + C2))
    
    return ssim_map.mean(dim=(1, 2, 3))

Writing ./sr/metrics.py


In [9]:
%%writefile ./sr/models.py
import torch
import torch.nn as nn
import torch.nn.functional as F

# JK - Channel Attention - https://arxiv.org/pdf/1807.02758
class ChannelAttention(nn.Module):
    def __init__(self, channels: int, reduction: int = 8):
        super().__init__()
        hidden = max(1, channels // reduction)
        self.net = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), 
            # For each image of dim=(H,W) using this "squeeze" activation level gets turned into 
            # a number. A block learns which channels are important for a given image.
            nn.Conv2d(channels, hidden, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.net(x)

# ----------------------------------------------------------

class ResidualBlock(nn.Module):
    """conv3x3 -> ReLU -> conv3x3, then add the input back."""

    def __init__(self, channels: int, attention: bool = False, reduction: int = 8):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding='same')
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding='same')
        self.attent = ChannelAttention(channels, reduction) if attention else nn.Identity() # Just a switch to enable / disable the CA mechanism.

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.conv2(F.relu(self.conv1(x))) # We're adding a residual connection here. And then using CA.
        y = self.attent(y)
        
        return x + y


class SuperResolutionNet(nn.Module):
    def __init__(self, scale: int = 2, channels: int = 64, n_blocks: int = 16, attention: bool = False, reduction: int = 8):
        super().__init__()
        self.scale = scale
        self.head = nn.Conv2d(3, channels, kernel_size=3, padding='same')
        self.blocks = nn.ModuleList([ResidualBlock(channels, attention, reduction) for _ in range(n_blocks)])
        self.to_rgb = nn.Conv2d(channels, 3 * self.scale ** 2, kernel_size=3, padding='same')

    def forward(self, lr: torch.Tensor) -> torch.Tensor:
        x = self.head(lr)
        for block in self.blocks:
            x = block(x)
        x = self.to_rgb(x)
        return F.pixel_shuffle(x, upscale_factor=self.scale)

def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
    
if __name__ == "__main__":
    for att in (False, True):
        m = SuperResolutionNet(scale=2, attention=att)
        y = m(torch.rand(2, 3, 48, 48))
        assert y.shape == (2, 3, 96, 96), y.shape
        print("attention =", att, f"{count_parameters(m):,} params")

Writing ./sr/models.py


In [10]:
%%writefile ./check.py

from pathlib import Path
import torch

print("1. model shapes")
from sr.models import SuperResolutionNet, count_parameters

for s in (2, 3, 4):
    y = SuperResolutionNet(scale=s, channels=16, n_blocks=2)(torch.rand(1, 3, 20, 24))
    assert y.shape == (1, 3, 20 * s, 24 * s), f"scale {s}: got {tuple(y.shape)}"
n = count_parameters(SuperResolutionNet(scale=2))
print(f"   ok, default model has {n:,} params")

print("2. metrics on synthetic data")
from sr.metrics import psnr, ssim

a = torch.randint(0, 256, (2, 3, 64, 64)) / 255.0  # 8-bit-valued images
assert torch.isinf(psnr(a, a)).all() or (psnr(a, a) > 90).all(), "identical images -> inf/very high PSNR"
assert torch.allclose(ssim(a, a).float(), torch.ones(2), atol=1e-6), "identical images -> SSIM 1"
assert psnr(a, a).shape == (2,), "psnr must return one value per image"
print("   ok")

print("3. data + bicubic baseline on Set5 (x2)")
from PIL import Image

from sr.data import SREvalDataset, SRTrainDataset, to_tensor

ds = SREvalDataset("data/Set5_HR", scale=2)
P = S = 0.0
for lr, hr, *_ in ds:
    assert hr.shape[-1] == 2 * lr.shape[-1] and hr.shape[-2] == 2 * lr.shape[-2]
    # upscale LR back with PIL bicubic as the reference method
    lr_img = Image.fromarray((lr.permute(1, 2, 0).numpy() * 255).round().astype("uint8"))
    bic = to_tensor(lr_img.resize((hr.shape[-1], hr.shape[-2]), Image.BICUBIC))
    P += psnr(bic[None], hr[None], crop=2).item()
    S += ssim(bic[None], hr[None], crop=2).item()
P, S = P / len(ds), S / len(ds)
print(f"   bicubic Set5 x2: PSNR {P:.2f} dB, SSIM {S:.4f}   (expected ~33.67 / ~0.930)")
assert abs(P - 33.67) < 0.15 and abs(S - 0.930) < 0.003, "metrics or degradation differ from the standard protocol"

tr = SRTrainDataset("data/Set5_HR", scale=2, lr_patch=48, crops_per_image=4)
lr, hr = tr[7]  # index > n_files also checks the idx % n_files wrap-around
assert lr.shape == (4, 3, 48, 48) and hr.shape == (4, 3, 96, 96), (lr.shape, hr.shape)
assert not torch.equal(hr[0], hr[1]), "crops from one image should differ"
print("   ok")
print("All checks passed.")


Writing ./check.py


In [11]:
%%writefile ./evaluate.py
import argparse
import csv
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # save figures without a display
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from PIL import Image

from sr.data import SREvalDataset, to_tensor
from sr.metrics import psnr, ssim
from sr.models import SuperResolutionNet, count_parameters


def load_checkpoint(path, device):
    """Rebuild the model from the saved hyper-parameters and load its weights."""
    checkpoint = torch.load(path, map_location=device)
    args = checkpoint["args"]
    model = SuperResolutionNet(scale=args["scale"], channels=args["channels"], n_blocks=args["n_blocks"])
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device).eval()
    return args["name"], model, args["scale"]


def bicubic_up(lr: torch.Tensor, scale: int) -> torch.Tensor:
    """Reference method: upscale LR back with PIL bicubic (same resampler as the degradation)."""
    img = Image.fromarray((lr.permute(1, 2, 0).numpy() * 255).round().astype("uint8"))
    return to_tensor(img.resize((img.width * scale, img.height * scale), Image.BICUBIC))


def most_detailed_box(hr: torch.Tensor, size: int) -> tuple[int, int]:
    """Top-left (y, x) of the size x size window with the most edges (sum of gradient magnitudes)."""
    gray = hr.mean(0, keepdim=True)[None]  # [1,1,H,W]
    grad = (gray[..., 1:, :-1] - gray[..., :-1, :-1]).abs() + (gray[..., :-1, 1:] - gray[..., :-1, :-1]).abs()
    stride = max(1, size // 4)
    energy = F.avg_pool2d(grad, size, stride=stride)[0, 0]
    idx = energy.flatten().argmax().item()
    return (idx // energy.shape[1]) * stride, (idx % energy.shape[1]) * stride


@torch.no_grad()
def evaluate_set(models, dataset, device, crop_dir: Path | None, crop_size: int):
    """Per-image PSNR/SSIM for bicubic and every model; optionally save crop figures."""
    scale = dataset.scale
    rows = []
    for lr, hr, name in dataset:
        outputs = {"Bicubic": bicubic_up(lr, scale).to(device)}
        for model_name, model, _ in models:
            outputs[model_name] = model(lr.unsqueeze(0).to(device)).clamp(0, 1)[0]

        hr_b = hr.unsqueeze(0).to(device)
        scores = {}
        for method, sr in outputs.items():
            p = psnr(sr.unsqueeze(0), hr_b, crop=scale).item()
            s = ssim(sr.unsqueeze(0), hr_b, crop=scale).item()
            scores[method] = p
            rows.append({"image": name, "method": method, "psnr": p, "ssim": s})

        if crop_dir is not None and min(hr.shape[-2:]) >= crop_size:
            outputs = {m: img.cpu() for m, img in outputs.items()}
            save_crop_figure(hr, outputs, scores, name, crop_dir, crop_size)
    return rows


def save_crop_figure(hr, outputs, scores, name, crop_dir: Path, size: int):
    y, x = most_detailed_box(hr, size)
    panels = [("HR", hr, None)] + [(m, img, scores[m]) for m, img in outputs.items()]
    fig, axes = plt.subplots(1, len(panels) + 1, figsize=(2.8 * (len(panels) + 1), 3.2))
    axes[0].imshow(hr.permute(1, 2, 0).numpy())
    axes[0].add_patch(plt.Rectangle((x, y), size, size, fill=False, edgecolor="red", linewidth=1.5))
    axes[0].set_title(name, fontsize=9)
    for ax, (title, img, p) in zip(axes[1:], panels):
        ax.imshow(img[:, y:y + size, x:x + size].permute(1, 2, 0).numpy(), interpolation="nearest")
        ax.set_title(title if p is None else f"{title}\n{p:.2f} dB", fontsize=9)
    for ax in axes:
        ax.axis("off")
    fig.tight_layout()
    crop_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(crop_dir / f"{name}.png", dpi=150)
    plt.close(fig)


def main():
    p = argparse.ArgumentParser()
    p.add_argument("checkpoints", nargs="+")
    p.add_argument("--sets", nargs="+", default=["data/Set5_HR", "data/Set14_HR", "data/DIV2K_valid_HR"])
    p.add_argument("--out", default="results")
    p.add_argument("--crop-sets", nargs="+", default=["data/Set5_HR", "data/Set14_HR"],
                   help="sets to save crop figures for (DIV2K valid would be 100 figures)")
    p.add_argument("--crop-size", type=int, default=96, help="zoom window size in HR pixels")
    args = p.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    models = [load_checkpoint(c, device) for c in args.checkpoints]  # load once, reuse for every set
    scales = {scale for _, _, scale in models}
    assert len(scales) == 1, f"all checkpoints must have the same scale, got {scales}"
    scale = scales.pop()
    params = {name: count_parameters(m) for name, m, _ in models}

    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    all_rows, summary = [], []
    for set_dir in args.sets:
        if not Path(set_dir).is_dir():
            print(f"skipping {set_dir} (not found)")
            continue
        set_name = Path(set_dir).name.removesuffix("_HR")
        dataset = SREvalDataset(set_dir, scale=scale)
        crop_dir = out / "crops" / set_name if set_dir in args.crop_sets else None
        rows = evaluate_set(models, dataset, device, crop_dir, args.crop_size)
        for r in rows:
            all_rows.append({"set": set_name, **r})
        for method in ["Bicubic"] + [name for name, _, _ in models]:
            mr = [r for r in rows if r["method"] == method]
            mean_psnr = sum(r["psnr"] for r in mr) / len(mr)
            mean_ssim = sum(r["ssim"] for r in mr) / len(mr)
            summary.append((set_name, method, params.get(method, "-"), mean_psnr, mean_ssim))
            print(f"{set_name:<12} {method:<16} PSNR {mean_psnr:.3f}  SSIM {mean_ssim:.4f}")

    with open(out / "metrics.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["set", "image", "method", "psnr", "ssim"])
        writer.writeheader()
        writer.writerows(all_rows)

    lines = [f"Scale x{scale}. PSNR/SSIM on Y channel, {scale} px border cropped.", "",
             "| Set | Method | Params | PSNR [dB] | SSIM |", "|---|---|---|---|---|"]
    for set_name, method, n, mp, ms in summary:
        n = f"{n:,}" if isinstance(n, int) else n
        lines.append(f"| {set_name} | {method} | {n} | {mp:.3f} | {ms:.4f} |")
    (out / "summary.md").write_text("\n".join(lines) + "\n")
    print("\n" + "\n".join(lines))
    print(f"\nSaved {out / 'summary.md'}, {out / 'metrics.csv'}, crops in {out / 'crops'}")


if __name__ == "__main__":
    main()


Writing ./evaluate.py


In [12]:
%%writefile ./train.py

import argparse
import csv
import time
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
import wandb

from sr.data import SREvalDataset, SRTrainDataset
from sr.metrics import psnr, ssim
from sr.models import SuperResolutionNet, count_parameters


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--name", required=True)
    p.add_argument("--train-dir", default="data/DIV2K_train_HR")
    p.add_argument("--val-dir", default="data/Set5_HR")
    p.add_argument("--scale", type=int, default=2)
    p.add_argument("--channels", type=int, default=64)
    p.add_argument("--n_blocks", type=int, default=16)
    p.add_argument("--iterations", type=int, default=100000)
    p.add_argument("--batch", type=int, default=16, help="patches per step")
    p.add_argument("--crops_per_image", type=int, default=8)
    p.add_argument("--lr", type=float, default=2e-4)
    p.add_argument("--lr_patch", type=int, default=48)
    p.add_argument("--loss", choices=["l1", "charbonnier"], default="l1") # Choice of Loss F.
    p.add_argument("--eps", type=float, default=1e-3) # Epsilon for loss function
    p.add_argument("--workers", type=int, default=16)
    p.add_argument("--val-every", type=int, default=1000)
    p.add_argument("--resume", action="store_true", help="continue from runs/<name>/last.pt")
    p.add_argument("--attention", action="store_true", help="add channel attention to residual blocks")
    p.add_argument("--reduction", type=int, default=8)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--run-id", type=str, default=None, help="W&B run ID to continue (default: the one saved in last.pt)")
    p.add_argument("--wandb", action="store_true", help="log to Weights & Biases")
    return p.parse_args()


def draw_PSNR(iterations, PSNR, outpath: Path):
    """Draw PSNR curves from the CSV log."""
    plt.figure(figsize=(8, 4))
    plt.plot(iterations, PSNR, label="PSNR")
    plt.xlabel("Iteration")
    plt.ylabel("PSNR (dB)")
    plt.title("PSNR (Validation)")
    plt.legend()
    plt.grid()
    plt.savefig(outpath)
    plt.close()



def draw_SSIM(iterations, SSIM, outpath: Path):
    """Draw SSIM curves from the CSV log."""
    plt.figure(figsize=(8, 4))
    plt.plot(iterations, SSIM, label="SSIM")
    plt.xlabel("Iteration")
    plt.ylabel("SSIM")
    plt.title("SSIM (Validation)")
    plt.legend()
    plt.grid()
    plt.savefig(outpath)
    plt.close()

def draw_losses(iterations, train_loss, val_loss, outpath: Path):
    """Draw training loss curve from the CSV log."""
    plt.figure(figsize=(8, 4))
    plt.plot(iterations, train_loss, label="Train Loss")
    plt.plot(iterations, val_loss, label="Validation Loss")
    plt.xlabel("Iteration")
    plt.ylabel("Loss")
    plt.title("Training Loss")
    plt.legend()
    plt.grid()
    plt.savefig(outpath)
    plt.close()


def analyze_training_log(log_path: Path):
    """Draw loss and accuracy curves from the CSV log."""
    iterations, train_loss, val_psnr, val_ssim, val_loss = [], [], [], [], []
    with open(log_path) as f:
        reader = csv.DictReader(f)
        for row in reader:
            iterations.append(int(row["iteration"]))
            train_loss.append(float(row["train_loss"]))
            val_psnr.append(float(row["val_psnr"]))
            val_ssim.append(float(row["val_ssim"]))
            val_loss.append(float(row["val_loss"]))
    out_dir = log_path.parent
    draw_PSNR(iterations, val_psnr, out_dir / "PSNR.png")
    draw_SSIM(iterations, val_ssim, out_dir / "SSIM.png")
    draw_losses(iterations, train_loss, val_loss, out_dir / "loss.png")

class CharbonnierLoss(torch.nn.Module):
    def __init__(self, eps: float = 1e-3):
        super().__init__()
        self.eps = eps

    def forward(self, sr, hr):
        return torch.sqrt((sr - hr) ** 2 + self.eps ** 2).mean()
    # We're adding Charbonnier Loss for checking out if this will bring us better results than MAE.

@torch.no_grad()
def validate(model, dataset, device) -> tuple[float, float]:
    model.eval()
    psnr_sum = 0.0
    ssim_sum = 0.0
    for lr, hr, _ in dataset:
        lr = lr.unsqueeze(0).to(device)
        hr = hr.unsqueeze(0).to(device)
        sr = model(lr).clamp(0,1)
        psnr_sum += psnr(sr, hr, crop=dataset.scale).item()
        ssim_sum += ssim(sr, hr, crop=dataset.scale).item()
    model.train()
    return psnr_sum / len(dataset), ssim_sum / len(dataset)


def main():
    args = parse_args()
    torch.manual_seed(args.seed)
    np.random.seed(args.seed) 

    assert args.batch % args.crops_per_image == 0, "--batch must be divisible by --crops_per_image"
    torch.backends.cudnn.benchmark = True  # fixed input size -> let cuDNN pick the fastest conv algorithm
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out = Path("runs") / args.name
    out.mkdir(parents=True, exist_ok=True)

    # Here we add the Attention to previously defined network:
    model = SuperResolutionNet(scale=args.scale, channels=args.channels, n_blocks=args.n_blocks, attention=args.attention, reduction=args.reduction).to(device)
    print(f"Model: {count_parameters(model):,} parameters")
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, betas=(0.9, 0.99))
    criterion = CharbonnierLoss(args.eps) if args.loss == "charbonnier" else torch.nn.L1Loss()
    # Use one of the two available loss functions with --loss --eps. 
    # Halve the LR at 50% and 75% of training, whatever --iterations is.
    milestones = [args.iterations // 2, args.iterations * 3 // 4]
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=milestones, gamma=0.5)

    iteration, best_psnr, run_id = 0, 0.0, None
    if args.resume and (out / "last.pt").exists():
        ckpt = torch.load(out / "last.pt", map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        iteration, best_psnr = ckpt["iteration"], ckpt["best_psnr"]
        run_id = ckpt.get("run_id")  # .get: older checkpoints have no run_id
        print(f"Resumed at iteration {iteration}, best PSNR {best_psnr:.3f}")
    if args.run_id:  # manual override
        run_id = args.run_id


    # After the checkpoint load, so a resumed run continues the same W&B run.
    run = None
    if not args.wandb:
        wandb.init(mode="disabled")
    else:
        run = wandb.init(
            project="super-resolution",
            name=args.name,
            config=vars(args),
            dir=out,  # W&B's local files go to runs/<name>/wandb/ (ignored by git)
            id=run_id,  # None -> new run
            resume="allow",
        )
        print(f"W&B run id: {run.id}")

    train_dataset = SRTrainDataset(hr_dir=args.train_dir, scale=args.scale, lr_patch=args.lr_patch,
                                   crops_per_image=args.crops_per_image)
    # Each dataset item is crops_per_image patches, so fewer items per batch.
    train_loader = DataLoader(train_dataset, batch_size=args.batch // args.crops_per_image, shuffle=True,
                              num_workers=args.workers, pin_memory=True, drop_last=True,
                              persistent_workers=True)
    val_dataset = SREvalDataset(hr_dir=args.val_dir, scale=args.scale)

    log_file = open(out / "log.csv", "a" if args.resume else "w", newline="")
    log = csv.writer(log_file)
    if not args.resume:
        log.writerow(["iteration", "train_loss", "lr", "val_psnr", "val_ssim"])

    def save(path):
        torch.save({
            "iteration": iteration,
            "best_psnr": best_psnr,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "run_id": run.id if run else None,
            "args": vars(args),
        }, path)

    loss_sum, t0 = 0.0, time.time()
    while iteration < args.iterations:  # iteration-based: re-iterate the loader as often as needed
        for lr, hr in train_loader:
            lr = lr.flatten(0, 1).to(device, non_blocking=True)  # [B,k,3,p,p] -> [B*k,3,p,p]
            hr = hr.flatten(0, 1).to(device, non_blocking=True)
            sr = model(lr)
            loss = criterion(sr, hr)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()
            iteration += 1
            loss_sum += loss.item()

            if iteration % args.val_every == 0 or iteration == args.iterations:
                n = args.val_every if iteration % args.val_every == 0 else iteration % args.val_every
                avg_loss = loss_sum / n
                val_psnr, val_ssim = validate(model, val_dataset, device)
                current_lr = optimizer.param_groups[0]["lr"]
                speed = n / (time.time() - t0)
                print(f"Iteration {iteration}/{args.iterations}  loss {avg_loss:.5f}  lr {current_lr:.1e}  "
                      f"val PSNR {val_psnr:.3f}  SSIM {val_ssim:.4f}  ({speed:.1f} it/s)")
                log.writerow([iteration, avg_loss, current_lr, val_psnr, val_ssim])
                log_file.flush()
                # wandb.Image expects uint8 in [0, 255] - it does NOT rescale [0, 1] floats (just casts -> black)
                img_hwc = sample.clamp(0, 1).mul(255).round().byte().permute(1, 2, 0).cpu().numpy()
                wandb.log({
                    "train/loss": avg_loss,
                    "val/psnr": val_psnr,
                    "val/ssim": val_ssim,
                    "val/loss": val_loss,
                    "lr": current_lr,
                    "val/sample": wandb.Image(img_hwc, caption=f"SR, iteration {iteration}"),
                }, step=iteration)
                if val_psnr > best_psnr:
                    best_psnr = val_psnr
                    save(out / "best.pt")
                save(out / "last.pt")
                loss_sum, t0 = 0.0, time.time()

            if iteration >= args.iterations:
                break

    log_file.close()
    print(f"Done. Best val PSNR {best_psnr:.3f} dB -> {out / 'best.pt'}")
    analyze_training_log(out / "log.csv")


if __name__ == "__main__":
    main()


Writing ./train.py


In [13]:
!python sr/models.py

attention = False 1,190,412 params
attention = True 1,207,948 params


In [14]:
!python check.py

1. model shapes
   ok, default model has 1,190,412 params
2. metrics on synthetic data
   ok
3. data + bicubic baseline on Set5 (x2)
   bicubic Set5 x2: PSNR 33.67 dB, SSIM 0.9303   (expected ~33.67 / ~0.930)
   ok
All checks passed.


In [15]:
import wandb

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("WANDB_API_KEY")
wandb.login(key=wandb_key)

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: konarskj (alexdad063-politechnika-warszawska) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [17]:
wandb.init()

CommError: the provided API key cannot access this resource, check credentials

In [ ]:
!python train.py --name with_attention --iterations 100 --lr 6e-4 --val-every 50 --workers 4 --loss charbonnier --attention --wandb

In [ ]:
from pathlib import Path
import torch
import torch.nn.functional as F
import torchtest2

import importlib, sr.models
importlib.reload(sr.models)
from sr.models import SuperResolutionNet

CKPT_FILENAME = "with_attention"

from sr.models import SuperResolutionNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt_path = Path(f"runs/{CKPT_FILENAME}/best.pt")
if not ckpt_path.exists():
    ckpt_path = Path(f"runs/{CKPT_FILENAME}/last.pt")

ckpt = torch.load(ckpt_path, map_location=device)
args = ckpt["args"]

model = SuperResolutionNet(
    scale=args["scale"],
    channels=args["channels"],
    n_blocks=args["n_blocks"],
    attention=args.get("attention", False), 
    reduction=args.get("reduction", 8)
).to(device)
model.load_state_dict(ckpt["model_state_dict"])

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=args["lr"],
    betas=(0.9, 0.99),
)
scale = args["scale"]
inputs = torch.rand(4, 3, 48, 48, device=device)
targets = torch.rand(4, 3, 48 * scale, 48 * scale, device=device)

report = torchtest2.check_model(
    model,
    inputs,
    targets,
    loss_fn=F.l1_loss,
    optimizer=optimizer,
    overfit_steps=500,
)
print(report)

In [ ]:
from pathlib import Path
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image

from sr.data import SREvalDataset, to_tensor
from sr.metrics import psnr, ssim
from sr.models import SuperResolutionNet


def upsample_pil(lr: torch.Tensor, scale: int, resample) -> torch.Tensor:
    arr = (lr.permute(1, 2, 0).detach().cpu().numpy().clip(0.0, 1.0) * 255.0).round().astype(np.uint8)
    img = Image.fromarray(arr)
    up = img.resize((img.width * scale, img.height * scale), resample)
    return to_tensor(up)


def pick_crop_box(
    hr: torch.Tensor,
    crop_size: int = 64,
    top: int | None = None,
    left: int | None = None,
) -> tuple[int, int, int]:
    h, w = hr.shape[-2], hr.shape[-1]
    sz = min(crop_size, h, w)
    if top is not None and left is not None:
        y = max(0, min(top, h - sz))
        x = max(0, min(left, w - sz))
        return y, x, sz

    gray = hr.mean(0, keepdim=True)[None]
    grad = (gray[..., 1:, :-1] - gray[..., :-1, :-1]).abs() + (
        gray[..., :-1, 1:] - gray[..., :-1, :-1]
    ).abs()
    stride = max(1, sz // 4)
    energy = F.avg_pool2d(grad, sz, stride=stride)[0, 0]
    idx = energy.flatten().argmax().item()
    y = min((idx // energy.shape[1]) * stride, h - sz)
    x = min((idx % energy.shape[1]) * stride, w - sz)
    return y, x, sz


@torch.no_grad()
def show_four_panels_crop(
    ckpt_path: str = f"runs/{CKPT_FILENAME}/best.pt",
    set_dir: str = "data/Set5_HR",
    index: int = 0,
    crop_size: int = 64,
    top: int | None = None,
    left: int | None = None,
    figsize: tuple[int, int] = (16, 4.5),
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    p = Path(ckpt_path)
    if not p.exists():
        p = Path(f"runs/{CKPT_FILENAME}/last.pt")

    ckpt = torch.load(p, map_location=device)
    cfg = ckpt["args"]
    scale = cfg["scale"]

    model = SuperResolutionNet(
        scale=scale,
        channels=cfg["channels"],
        n_blocks=cfg["n_blocks"],
        attention=cfg.get("attention", False),
        reduction=cfg.get("reduction", 8),
    ).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    ds = SREvalDataset(set_dir, scale=scale)
    lr, hr, name = ds[index]

    lr_b = lr.unsqueeze(0).to(device)
    hr_b = hr.unsqueeze(0).to(device)

    lr_nearest = upsample_pil(lr, scale=scale, resample=Image.NEAREST)
    bicubic = upsample_pil(lr, scale=scale, resample=Image.BICUBIC)
    recon = model(lr_b).clamp(0.0, 1.0)[0].cpu()

    p_bic = psnr(bicubic.unsqueeze(0).to(device), hr_b, crop=scale).item()
    s_bic = ssim(bicubic.unsqueeze(0).to(device), hr_b, crop=scale).item()

    p_rec = psnr(recon.unsqueeze(0).to(device), hr_b, crop=scale).item()
    s_rec = ssim(recon.unsqueeze(0).to(device), hr_b, crop=scale).item()

    y, x, sz = pick_crop_box(hr, crop_size=crop_size, top=top, left=left)

    panels = [
        (f"LR (nearest) [{y}:{y+sz}, {x}:{x+sz}]\n{name}", lr_nearest),
        (f"Bicubic (base)\nPSNR: {p_bic:.2f} dB | SSIM: {s_bic:.4f}", bicubic),
        (f"Rekonstrukcja ({p.parent.name})\nPSNR: {p_rec:.4f} dB | SSIM: {s_rec:.4f}", recon),
        ("HR", hr),
    ]

    fig, axes = plt.subplots(1, 4, figsize=figsize)
    for ax, (title, img) in zip(axes, panels):
        patch = img[:, y : y + sz, x : x + sz].permute(1, 2, 0).detach().cpu().numpy().clip(0.0, 1.0)
        ax.imshow(patch, interpolation="nearest")
        ax.set_title(title, fontsize=10)
        ax.axis("off")

    plt.tight_layout()
    display(fig)
    plt.savefig(f"./runs/{CKPT_FILENAME}/comparison.png")
    plt.close(fig)


show_four_panels_crop(index=0, crop_size=64)

In [ ]:
!zip -r /kaggle/working/runs/no_attention.zip /kaggle/working/runs/no_attention
!zip -r /kaggle/working/runs/with_attention.zip /kaggle/working/runs/with_attention